# 1.3 非奇异矩阵与数值计算

本 notebook 回答两个问题：

- 矩阵可逆时，计算结果是否一定可靠？
- 用历史数据算出的样本协方差矩阵是否一定可逆？

## 问题 1：已知 $A$ 和 $b$，怎样用 Python 求解 $Ax=b$？

先看一个容易求解的方程组：

$$
\begin{cases}x_1+x_2=5,\\x_1-x_2=1.\end{cases}
\qquad
A=\begin{pmatrix}1&1\\1&-1\end{pmatrix},\quad
b=\begin{pmatrix}5\\1\end{pmatrix}.
$$

它的解是 $x=(3,2)^T$。数学上可以写成 $x=A^{-1}b$；实际代码直接使用 `np.linalg.solve(A, b)`，不必计算逆矩阵。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

A = np.array([[1.0, 1.0], [1.0, -1.0]])
b = np.array([5.0, 1.0])
x = np.linalg.solve(A, b)

np.testing.assert_allclose(x, [3.0, 2.0])
np.testing.assert_allclose(A @ x, b)

print("解 x：", x)
print("代回 A @ x：", A @ x)

解 x： [3. 2.]
代回 A @ x： [5. 1.]


`solve` 直接求所需的解，通常比 `np.linalg.inv(A) @ b` 更省计算，也避免显式求逆产生的额外数值误差。但 `solve` 不能消除方程本身对数据误差的敏感性。下面看一个反例。

## 问题 2：矩阵可逆，为什么数据略有变化，解却变化很大？

考虑

$$
A=\begin{pmatrix}1&1\\1&1.000001\end{pmatrix},\qquad
b=\begin{pmatrix}2\\2.000001\end{pmatrix}.
$$

两条方程相减，得到 $0.000001x_2=0.000001$，所以 $x_2=1$，$x_1=1$。它的行列式为 $0.000001\ne0$，因此可逆。

现在仅把 $b$ 的第二项改为 $2.000002$。相减后变成 $0.000001x_2=0.000002$，所以 $x_2=2$、$x_1=0$。输入只差百万分之一，解却明显改变。

In [5]:
A1 = np.array([[1.0, 1.0], [1.0, 1.000001]])
b1 = np.array([2.0, 2.000001])
b2 = np.array([2.0, 2.000002])
x1 = np.linalg.solve(A1, b1)
x2 = np.linalg.solve(A1, b2)

display(pd.DataFrame({
    "原始解": x1,
    "改变 b 后的解": x2,
}, index=["x1", "x2"]))

np.testing.assert_allclose(x1, [1.0, 1.0], atol=1e-8)
np.testing.assert_allclose(x2, [0.0, 2.0], atol=1e-8)

,原始解,改变 b 后的解
x1,1.0,0.0
x2,1.0,2.0


当 $A$ 接近奇异时，$A$ 或 $b$ 中很小的误差，也可能引起解的明显变化。因此，即使用 solve 得到了唯一解，也需要结合输入数据的精度判断结果是否可靠。